# CBC additional robustness analysis plotting-only RUN ALL

Notebook ini **khusus untuk membuat gambar/figure dari hasil additional analysis yang sudah selesai**.
Notebook ini **tidak melatih model lagi** dan **tidak mengubah running lama**.

Tujuan notebook ini adalah memberi **bukti yang bisa ditunjukkan ke dosen** bahwa gambar memang dibuat dengan proses yang dapat dirun ulang langsung dari file hasil (`CBC_ADDITIONAL ANALYSIS_RESULTS_FOR_REVIEW.zip`).

Output:
- 4 figure (`.png` dan `.pdf`)
- source CSV untuk figure
- log/manifest pembuatan figure
- ZIP final `CBC_ADDITIONAL ANALYSIS_PLOTTING_OUTPUT.zip`

Silakan pilih **Runtime → Run all**.


In [ ]:
# 1) Mount Drive and locate the results ZIP
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
import zipfile, shutil, sys

MYDRIVE = Path('/content/drive/MyDrive')
CONTENT = Path('/content')

def newest(paths):
    paths = [Path(p) for p in paths if Path(p).exists()]
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None

cands = (
    list(MYDRIVE.rglob('CBC_ADDITIONAL ANALYSIS_RESULTS_FOR_REVIEW.zip')) +
    list(MYDRIVE.rglob('*CBC_ADDITIONAL ANALYSIS*RESULTS*REVIEW*.zip')) +
    list(CONTENT.glob('CBC_ADDITIONAL ANALYSIS_RESULTS_FOR_REVIEW.zip')) +
    list(CONTENT.glob('*CBC_ADDITIONAL ANALYSIS*RESULTS*REVIEW*.zip'))
)
RESULTS_ZIP = newest(cands)
if RESULTS_ZIP is None:
    raise FileNotFoundError('CBC_ADDITIONAL ANALYSIS_RESULTS_FOR_REVIEW.zip tidak ditemukan. Upload ke MyDrive lalu Run all lagi.')

WORK = (MYDRIVE / 'CBC_ADDITIONAL ANALYSIS_PLOTTING_OUTPUT_20260830').resolve()
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

EXTRACT = WORK / 'unzipped_results'
EXTRACT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(RESULTS_ZIP, 'r') as z:
    z.extractall(EXTRACT)

print('Results ZIP:', RESULTS_ZIP)
print('Workspace  :', WORK)


In [ ]:
# 2) Install/import plotting dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas>=2,<3', 'matplotlib>=3.8,<4'])
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
print('Dependencies ready.')


In [ ]:
# 3) Load additional analysis CSV outputs
BASE = EXTRACT
strict_repeat = pd.read_csv(BASE / '02_repeated_fold_fitted_5x5' / 'repeated_fold_fitted_5x5_stability_by_repeat.csv')
strict_summary = pd.read_csv(BASE / '02_repeated_fold_fitted_5x5' / 'repeated_fold_fitted_5x5_stability_summary.csv')
ss_repeat = pd.read_csv(BASE / '03_stability_selection_repeated_10x5' / 'all_methods_plus_stability_selection_stability_by_repeat.csv')
ss_summary = pd.read_csv(BASE / '03_stability_selection_repeated_10x5' / 'all_methods_plus_stability_selection_stability_summary.csv')
auc = pd.read_csv(BASE / '03_stability_selection_repeated_10x5' / 'reported_existing_plus_stability_selection_lr_auc.csv')
boot = pd.read_csv(BASE / '01_gse15852_paired' / 'GSE15852_pair_cluster_bootstrap_distribution.csv.gz')
gse_auc = pd.read_csv(BASE / '01_gse15852_paired' / 'GSE15852_pair_cluster_bootstrap_auc.csv')

FIGDIR = WORK / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)
print('CSV inputs loaded.')


In [ ]:
# 4) Figure A — repeated strict training-fold-fitted stability (5×5-fold)
plt.figure(figsize=(8.2, 5.6))
for method, g in strict_repeat.groupby('Method', sort=False):
    g = g.sort_values('Repeat')
    plt.plot(g['Repeat'], g['Mean_Jaccard'], marker='o', linewidth=1.8, label=method)
plt.xticks(sorted(strict_repeat['Repeat'].unique()))
plt.ylim(0, 1.0)
plt.xlabel('Repeat')
plt.ylabel('Mean within-repeat pairwise Jaccard')
plt.title('Repeated strict training-fold-fitted stability (5×5-fold)')
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGDIR / 'Figure_A_Strict_FoldFitted_5x5_Jaccard.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGDIR / 'Figure_A_Strict_FoldFitted_5x5_Jaccard.pdf', bbox_inches='tight')
plt.show()
plt.close()
strict_repeat.to_csv(FIGDIR / 'Figure_A_source_data.csv', index=False)


In [ ]:
# 5) Figure B — repeated 10×5-fold with stability-selection comparator
plt.figure(figsize=(8.6, 5.8))
for method, g in ss_repeat.groupby('Method', sort=False):
    g = g.sort_values('Repeat')
    plt.plot(g['Repeat'], g['Mean_Jaccard'], marker='o', linewidth=1.6, label=method)
plt.xticks(sorted(ss_repeat['Repeat'].unique()))
plt.ylim(0, 1.0)
plt.xlabel('Repeat')
plt.ylabel('Mean within-repeat pairwise Jaccard')
plt.title('Repeated 10×5-fold stability with stability-selection comparator')
plt.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.savefig(FIGDIR / 'Figure_B_StabilitySelection_Comparator_10x5.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGDIR / 'Figure_B_StabilitySelection_Comparator_10x5.pdf', bbox_inches='tight')
plt.show()
plt.close()
ss_repeat.to_csv(FIGDIR / 'Figure_B_source_data.csv', index=False)


In [ ]:
# 6) Figure C — stability–discrimination trade-off
trade = ss_summary[['Method', 'Mean_Jaccard']].merge(
    auc[['Method', 'LR_OOF_ROC_AUC_mean']], on='Method', how='inner'
)
plt.figure(figsize=(7.6, 5.8))
plt.scatter(trade['Mean_Jaccard'], trade['LR_OOF_ROC_AUC_mean'], s=70)
for _, r in trade.iterrows():
    plt.annotate(r['Method'], (r['Mean_Jaccard'], r['LR_OOF_ROC_AUC_mean']),
                 xytext=(6, 5), textcoords='offset points', fontsize=9)
plt.xlabel('Repeated mean Jaccard stability')
plt.ylabel('LR out-of-fold ROC-AUC')
plt.title('Stability–discrimination trade-off under repeated 10×5-fold evaluation')
plt.tight_layout()
plt.savefig(FIGDIR / 'Figure_C_Stability_vs_LR_AUC.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGDIR / 'Figure_C_Stability_vs_LR_AUC.pdf', bbox_inches='tight')
plt.show()
plt.close()
trade.to_csv(FIGDIR / 'Figure_C_source_data.csv', index=False)


In [ ]:
# 7) Figure D — GSE15852 paired bootstrap distribution
order = ['LR', 'RF', 'LightGBM']
data = [boot.loc[boot['Classifier'] == m, 'ROC_AUC'].dropna().values for m in order]
plt.figure(figsize=(7.4, 5.6))
plt.boxplot(data, tick_labels=order, showfliers=False)
for i, m in enumerate(order, start=1):
    point = float(gse_auc.loc[gse_auc['Classifier'] == m, 'ROC_AUC'].iloc[0])
    plt.scatter(i, point, marker='D', s=45, label='Observed ROC-AUC' if i == 1 else None)
plt.ylim(0.6, 1.0)
plt.xlabel('Classifier')
plt.ylabel('ROC-AUC')
plt.title('GSE15852: 2,000 patient-pair cluster bootstrap replicates')
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(FIGDIR / 'Figure_D_GSE15852_PairedBootstrap.png', dpi=300, bbox_inches='tight')
plt.savefig(FIGDIR / 'Figure_D_GSE15852_PairedBootstrap.pdf', bbox_inches='tight')
plt.show()
plt.close()
gse_auc.to_csv(FIGDIR / 'Figure_D_point_estimates_and_CI.csv', index=False)


In [ ]:
# 8) Save manifest, captions, and final ZIP
import hashlib, json, os
manifest = {
    'created_utc': __import__('datetime').datetime.utcnow().isoformat() + 'Z',
    'results_zip': str(RESULTS_ZIP),
    'output_dir': str(FIGDIR),
    'figures': sorted([p.name for p in FIGDIR.glob('*.png')]),
}
for p in sorted(FIGDIR.iterdir()):
    if p.is_file():
        manifest[p.name] = {
            'size_bytes': p.stat().st_size,
            'sha256': hashlib.sha256(p.read_bytes()).hexdigest(),
        }
with open(FIGDIR / 'PLOTTING_MANIFEST.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

captions = '''# Figure captions and proof note

Notebook ini adalah **bukti run** bahwa figure additional analysis dibuat langsung dari file hasil final.
Tidak ada training model ulang di notebook plotting ini.

- Figure A: repeated strict training-fold-fitted stability (5×5-fold).
- Figure B: repeated 10×5-fold stability with stability-selection comparator.
- Figure C: stability–discrimination trade-off.
- Figure D: GSE15852 paired bootstrap distribution.

Sumber data figure:
- `02_repeated_fold_fitted_5x5/repeated_fold_fitted_5x5_stability_by_repeat.csv`
- `03_stability_selection_repeated_10x5/all_methods_plus_stability_selection_stability_by_repeat.csv`
- `03_stability_selection_repeated_10x5/all_methods_plus_stability_selection_stability_summary.csv`
- `03_stability_selection_repeated_10x5/reported_existing_plus_stability_selection_lr_auc.csv`
- `01_gse15852_paired/GSE15852_pair_cluster_bootstrap_distribution.csv.gz`
- `01_gse15852_paired/GSE15852_pair_cluster_bootstrap_auc.csv`
'''
with open(FIGDIR / 'FIGURE_CAPTIONS_AND_PROOF_NOTE.md', 'w', encoding='utf-8') as f:
    f.write(captions)

ZIP_OUT = WORK / 'CBC_ADDITIONAL ANALYSIS_PLOTTING_OUTPUT.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in FIGDIR.rglob('*'):
        if p.is_file():
            z.write(p, arcname=str(Path('CBC_ADDITIONAL ANALYSIS_PLOTTING_OUTPUT') / p.relative_to(FIGDIR)))

print('DONE.')
print('Figure folder:', FIGDIR)
print('ZIP final    :', ZIP_OUT)
